In [ ]:
%pip install statsbombpy

In [1]:
import pandas as pd
import numpy as np

In [2]:
from statsbombpy import sb

competitions = sb.competitions()

competition_cols = ["competition_id",
            "competition_name",
            "season_id",
            "season_name",]

# competitions w/ data available for 2015/2016
print(competitions[competitions["season_id"] == 27][competition_cols])

competitions_15_16 = competitions[competitions["season_id"] == 27][competition_cols]

top_league_comp_ids = dict(zip(competitions_15_16["competition_name"], competitions_15_16["competition_id"]))

/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


    competition_id  competition_name  season_id season_name
1                9     1. Bundesliga         27   2015/2016
6               16  Champions League         27   2015/2016
45              11           La Liga         27   2015/2016
63               7           Ligue 1         27   2015/2016
68               2    Premier League         27   2015/2016
70              12           Serie A         27   2015/2016


In [4]:
top_league_comp_ids

{'1. Bundesliga': 9,
 'Champions League': 16,
 'La Liga': 11,
 'Ligue 1': 7,
 'Premier League': 2,
 'Serie A': 12}

In [3]:
match_cols = ["match_id",
            "match_date",
            "home_team",
            "away_team",]
match_df = pd.DataFrame()

for comp_name, comp_id in top_league_comp_ids.items():
    
    matches = sb.matches(
        competition_id=comp_id,
        season_id=27,
    )

    print(comp_name + ": \n")
    print(matches.shape)

    if len(matches) > 300:
        match_df = pd.concat([match_df, matches[match_cols]], ignore_index=True)

match_df.head(10)

/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


1. Bundesliga: 

(34, 55)
Champions League: 

(1, 55)


/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


La Liga: 

(380, 55)


/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


...

/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Premier League: 

(380, 55)


/opt/anaconda3/envs/advML/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Serie A: 

(380, 55)


,match_id,match_date,home_team,away_team
0,3825739,2016-01-17,Real Madrid,Sporting Gijón
1,3825848,2015-09-23,Levante UD,Eibar
2,3825895,2015-09-23,Las Palmas,Sevilla
3,3825894,2016-05-01,RC Deportivo La Coruña,Getafe
4,3825855,2016-05-02,Málaga,Levante UD
5,3825908,2016-05-15,Espanyol,Eibar
6,3825883,2016-05-15,Málaga,Las Palmas
7,3825900,2016-05-15,Sporting Gijón,Villarreal
8,3825902,2016-05-15,Rayo Vallecano,Levante UD
9,3825876,2016-05-15,Real Betis,Getafe


In [ ]:
print(pd.to_datetime(match_df["match_date"]).describe())

print(match_df["home_team"].value_counts())

In [ ]:
match_id = matches.iloc[0]["match_id"]

events = sb.events(match_id=match_id)

print(events.columns.tolist())

#events = events[events["player"].notnull()]

print(
    events[
        [
            "minute",
            "team",
            "player",
            "type",
            "location",
        ]
    ].head()
)
print(events[events["player"].isnull()]["type"].value_counts())
print(events[events["type"] == "Substitution"]["player"])
print(events["player"].value_counts())

In [ ]:
for match_id in match_df["match_id"].head(5).tolist():

    events = sb.events(match_id=match_id)

    print(events.shape)



In [ ]:
matches = sb.matches(
        competition_id=2,
        season_id=27,
    )

matches[["home_team", "away_team"]].value_counts()
print(len(matches))

In [ ]:
import os

os.chdir("..")

print(os.getcwd())

from src.ingest import ingest_matches

ingest_matches(competition_id=2, season_id=27)

In [ ]:
from pathlib import Path

data_dir = Path("data/raw/Premier League/2015-2016")

files = list(data_dir.glob("*.parquet"))

events = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=False
)

events.columns.tolist()

In [ ]:
len(events)

# Exploring different event types #

In [ ]:
events["type"].value_counts()

In [1]:
from pathlib import Path

competition = "Premier League"

season = "2015-2016"

project_dir = Path.cwd()

processed_dir = project_dir / "data/processed" / competition / season

## Passes ##

In [ ]:
passes = events[events["type"] == "Pass"]

passes.dropna(axis=1, how="all").columns.tolist()

In [ ]:
passes.dropna(axis=1, how="all").head(5)

In [ ]:
pd.set_option('display.max_columns', None)

pass_path = processed_dir / "passes.parquet"
proc_passes = pd.read_parquet(pass_path)

proc_passes.head(10)

## Dribbles ##

In [ ]:
dribbles = events[events["type"] == "Dribble"]

dribbles.dropna(axis=1, how="all").columns.tolist()

In [ ]:
dribbles.dropna(axis=1, how="all").head(5)

## Carries ##

In [ ]:
carries = events[events["type"] == "Carry"]

carries.dropna(axis=1, how="all").columns.tolist()

In [ ]:
carries.dropna(axis=1, how="all").head(5)

## Shots ##

In [ ]:
shots = events[events["type"] == "Shot"]

shots.dropna(axis=1, how="all").columns.tolist()

In [ ]:
#shots.dropna(axis=1, how="all")[["shot_statsbomb_xg"]].head(5)
shots.dropna(axis=1, how="all").dtypes

In [ ]:
shots.dropna(axis=1, how="all").head(5)

## Interceptions ##

In [ ]:
interceptions = events[events["type"] == "Interception"]

interceptions.dropna(axis=1, how="all").columns.tolist()

In [ ]:
interceptions.dropna(axis=1, how="all").head(5)

In [ ]:
interceptions[["interception_outcome"]].value_counts()

## Pressures ##

In [ ]:
pressures = events[events["type"] == "Pressure"]

pressures.dropna(axis=1, how="all").columns.tolist()

In [ ]:
pressures.dropna(axis=1, how="all").head(5)

In [ ]:
pressure_path = processed_dir / "pressures.parquet"
proc_pressures = pd.read_parquet(pressure_path)

pressure_features = (
    proc_pressures
    .groupby(["match_id", "player_id"])
    .size()
    .reset_index(name="pressures")
)

## Duels ##

In [ ]:
duels = events[events["type"] == "Duel"]

duels.dropna(axis=1, how="all").columns.tolist()

In [ ]:
print(duels["duel_type"].value_counts(dropna=False))

print()

print(duels["duel_outcome"].value_counts(dropna=False))

In [ ]:
pd.crosstab(

    duels["duel_type"],

    duels["duel_outcome"],

    dropna=False

)

## Clearances ##

In [ ]:
clearances = events[events["type"] == "Clearance"]

clearances.dropna(axis=1, how="all").columns.tolist()

## Blocks ##

In [ ]:
blocks = events[events["type"] == "Block"]

blocks.dropna(axis=1, how="all").columns.tolist()

In [ ]:
blocks[[
    "block_offensive",
    "block_deflection",
    "block_save_block"
]].value_counts(dropna=False)

## Ball Recoveries ##

In [ ]:
ball_recoveries = events[events["type"] == "Ball Recovery"]

ball_recoveries.dropna(axis=1, how="all").columns.tolist()

In [ ]:
ball_recoveries.dropna(axis=1, how="all").head(5)

In [ ]:
recoveries = events[
    events["type"] == "Ball Recovery"
]

recoveries[
    "ball_recovery_recovery_failure"
].value_counts(dropna=False)

## Fouls Committed ##

In [ ]:
fouls = events[events["type"] == "Foul Committed"]

fouls.dropna(axis=1, how="all").columns.tolist()

In [ ]:
fouls.dropna(axis=1, how="all").head(5)

## Figuring out minutes ##

In [ ]:
minutes = events[
    events["type"].isin([
        "Player On",
        "Player Off",
        "Substitution",
        "Half End",
        "Half Start",
        "Starting XI",
        "Bad Behaviour"
    ])
].copy()

minutes.dropna(axis=1, how="all").columns.tolist()

In [ ]:
print(
    events["bad_behaviour_card"]
    .dropna()
    .value_counts()
)

print(
    events["foul_committed_card"]
    .dropna()
    .value_counts()
)

In [ ]:
pd.set_option('display.max_columns', None)
minutes.dropna(axis=1, how="all").head(25)

In [ ]:
minutes.dropna(axis=1, how="all").iloc[0]["tactics"]

In [ ]:
def validate_player_minutes(player_minutes):

    """
    Runs sanity checks on reconstructed player participation intervals.
    """

    # 1. Each player should have at most one record per match.
    duplicates = player_minutes[
        player_minutes.duplicated(
            subset=["match_id", "player_id"],
            keep=False
        )
    ]

    print(f"Duplicate player-match records: {len(duplicates)}")

    # 2. No player should enter before the match begins.
    invalid_time_on = player_minutes[
        player_minutes["time_on"] < 0
    ]

    print(f"Negative time_on values: {len(invalid_time_on)}")

    # 3. No player should leave before they entered.
    invalid_intervals = player_minutes[
        player_minutes["time_off"] < player_minutes["time_on"]
    ]

    print(f"time_off < time_on: {len(invalid_intervals)}")

    # 4. Minutes played should never be negative.
    negative_minutes = player_minutes[
        player_minutes["minutes_played"] < 0
    ]

    print(f"Negative minutes played: {len(negative_minutes)}")

    # 5. Starters should always have time_on == 0.
    invalid_starters = player_minutes[
        player_minutes["started"]
        & (player_minutes["time_on"] != 0)
    ]

    print(f"Starters with non-zero time_on: {len(invalid_starters)}")

    # 6. Substitutes should normally have time_on > 0.
    invalid_subs = player_minutes[
        ~player_minutes["started"]
        & (player_minutes["time_on"] <= 0)
    ]

    print(f"Substitutes with time_on <= 0: {len(invalid_subs)}")

    # 7. Required fields should not be null.
    required_columns = [
        "match_id",
        "player_id",
        "team_id",
        "time_on",
        "time_off",
        "minutes_played",
    ]

    null_counts = (
        player_minutes[required_columns]
        .isna()
        .sum()
    )

    print("\nNull values:")
    print(null_counts)

    return {
        "duplicates": duplicates,
        "invalid_time_on": invalid_time_on,
        "invalid_intervals": invalid_intervals,
        "negative_minutes": negative_minutes,
        "invalid_starters": invalid_starters,
        "invalid_subs": invalid_subs,
    }

## Test ##

In [ ]:
import os
os.chdir("..")
print(os.getcwd())
from src.clean import clean_season

clean_season(competition_id=2, season_id=27)

In [2]:
import os
os.chdir("..")
print(os.getcwd())
from src.features import build_player_match_features

build_player_match_features(competition_id=2, season_id=27)

/Users/TeddyKolios/personal proj/statsbomb
Building features for competition: Premier League, season: 2015-2016....
Aggregating pass features...
Aggregating dribble features...
Aggregating carry features...
Aggregating shot features...
Aggregating duel features...
Aggregating clearance features...
Aggregating block features...
Aggregating interception features...
Aggregating ball recovery features...
Aggregating pressure features...
Aggregating foul features...


In [1]:
def sanity_check_player_match_features(processed_dir):
    """
    Run basic sanity checks on the player-match feature table.
    """
    features_path = processed_dir / "player_match_features.parquet"
    df = pd.read_parquet(features_path)

    print(f"Rows: {len(df):,}")
    print(f"Matches: {df['match_id'].nunique():,}")
    print(f"Players: {df['player_id'].nunique():,}")
    print(f"Columns: {len(df.columns)}")

    # Each player should appear at most once per match
    duplicates = df.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
    assert duplicates == 0, (
        f"Found {duplicates} duplicate player-match rows."
    )

    # IDs and minutes should always be present
    required_columns = ["match_id", "player_id", "minutes_played"]
    missing_required = df[required_columns].isna().sum()
    assert missing_required.sum() == 0, (
        f"Missing required values:\n{missing_required}"
    )

    # Event features should not contain missing values
    feature_columns = [
        col for col in df.columns
        if col not in required_columns
    ]
    missing_features = df[feature_columns].isna().sum()
    assert missing_features.sum() == 0, (
        f"Missing feature values:\n"
        f"{missing_features[missing_features > 0]}"
    )

    # Count and xG/xA features should never be negative
    numeric_features = df[feature_columns].select_dtypes(
        include="number"
    ).columns
    negative_counts = (df[numeric_features] < 0).sum()
    assert negative_counts.sum() == 0, (
        f"Negative feature values:\n"
        f"{negative_counts[negative_counts > 0]}"
    )

    # Basic passing relationships
    assert (
        df["passes_completed"] <= df["passes_attempted"]
    ).all(), "Found completed passes > attempted passes."
    assert (
        df["progressive_passes"] <= df["passes_completed"]
    ).all(), "Found progressive passes > completed passes."

    # Basic dribbling relationship
    assert (
        df["dribbles_completed"] <= df["dribbles_attempted"]
    ).all(), "Found completed dribbles > attempted dribbles."

    # Basic carry relationship
    assert (
        df["progressive_carries"] <= df["carries"]
    ).all(), "Found progressive carries > total carries."

    # Basic shooting relationships
    assert (
        df["shots_on_target"] <= df["shots"]
    ).all(), "Found shots on target > total shots."
    assert (
        df["goals"] <= df["shots_on_target"]
    ).all(), "Found goals > shots on target."

    # Chance creation relationships
    assert (
        df["assists"] <= df["shot_assists"]
    ).all(), "Found assists > shot assists."

    print("All sanity checks passed.")

In [2]:
import os
import pandas as pd
import numpy as np
os.chdir("..")
print(os.getcwd())

from pathlib import Path
competition = "Premier League"
season = "2015-2016"
project_dir = Path.cwd()
processed_dir = project_dir / "data/processed" / competition / season

sanity_check_player_match_features(processed_dir)

/Users/TeddyKolios/personal proj/statsbomb
Rows: 10,469
Matches: 380
Players: 550
Columns: 36
All sanity checks passed.
